# Codesign with Vitis


In [ ]:
# Make this notebook self-sufficient regardless of how the Jupyter kernel was
# launched. Some launchers (e.g. VS Code's "Python Environments" kernel picker)
# start the conda env's python directly, bypassing the custom kernel.json that
# would otherwise source Vitis HLS and disable user site-packages - so do both
# explicitly here, in-process, instead of relying on the launch environment.
import os
import sys
import site
import subprocess
import shutil

# Drop ~/.local site-packages: conda's python leaves user-site enabled by
# default, which can shadow this env's own (version-matched) qkeras/tf_keras
# with an incompatible one installed under --user.
_user_site = site.getusersitepackages()
sys.path = [p for p in sys.path if p != _user_site]

# Bring in the Vitis HLS 2024.2 environment (PATH, XILINX_HLS, license vars, ...)
_vitis_settings = "/extras2/tools/Xilinx/Vitis_HLS/2024.2/settings64.sh"
_env_dump = subprocess.run(
    f"source {_vitis_settings} && env -0",
    shell=True, executable="/bin/bash", capture_output=True, check=True,
).stdout
for _entry in _env_dump.decode().split("\0"):
    if _entry:
        _key, _, _value = _entry.partition("=")
        os.environ[_key] = _value

print("vitis_hls:", shutil.which("vitis_hls"))


## List trained models

In [ ]:
# Models trained on the ROI dataset (two_bit_optimization.ipynb with use_roi=True, roi_dataset='2s').
# Each directory name encodes the model_type and its fingerprint, both needed below.
!ls -d smart-pixels/weights/dataset_2s_roi_weights/*-2bit_optimized-*-checkpoints

## Setup

This notebook requires the `smartpixels-regression-vitis-env` conda environment. Set it up once with:

```
conda env create -f environment_vitis.yml
```

Then select the **"Vitis (smartpixels-regression-vitis-env)"** kernel (or any kernel using that env's Python interpreter). The first code cell below sources Vitis HLS 2024.2 and fixes up `sys.path` itself, so it works regardless of how the kernel was launched - no need to manually `source envsetup_vitis.sh` first. (`envsetup_vitis.sh` / `runme_vitis.sh` are still there for launching JupyterLab from the command line the same way `envsetup.sh` / `runme.sh` do for the Catapult notebook.)


In [ ]:
# Path to the Vitis HLS installation directory
!command -v vitis_hls


In [ ]:
# Vitis HLS version in use
!vitis_hls -version


In [ ]:
# For non-quantized models: Conv2D_Max, Conv2D_Full, Conv2D_Slim, Conv1D_Full, Conv1D_Slim, Mlp_Full, Mlp_Slim
# For quantized models: QConv2D_Max, QConv2D_Full, QConv2D_Slim, QConv1D_Full, QConv1D_Slim, QMlp_Full, QMlp_Slim
model_type = 'QMlp_Slim' 

## Import libraries

Disable some console warnings on the ASIC-group servers

In [ ]:
import os

In [ ]:
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [ ]:
import sys
sys.path.insert(0, "./two_bit_optimization_helpers")  # run this notebook from the repo root

In [ ]:
import hls4ml
# submodule is not pulled in by "import hls4ml" alone; cell using get_ymodel_keras needs it
import hls4ml.model.profiling

import numpy as np
import math
import tensorflow as tf
import yaml
import json
from two_bit_optimization_helpers import *

from matplotlib import colors
import matplotlib.pyplot as plt

from two_bit_optimization_helpers.loss import *
from qkeras import quantized_bits

from two_bit_optimization_helpers.codesign_utils import *

In [ ]:
!ls /extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/implementation/testbench/cocotb/release/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/

## Set parameters

In [ ]:
# ROI dataset produced by the VIAS roi_finder testbench (labels corrected, see VIAS commit 248aeed)
dataset_base_dir = "/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/implementation/testbench/cocotb/release/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/"

# .npy inputs are written by save_data_as_npy() at the end of two_bit_optimization.ipynb
npy_dir_val = os.path.join("npy")

# NOTE: tfrecords_dir_train/val used to be defined here but were never read anywhere in this
# notebook - the model is loaded from HDF5 and the data from .npy - so they were removed.

In [ ]:
# batch_size = 5000
# train_file_size = len(os.listdir(dataset_train_dir))
# val_file_size = len(os.listdir(dataset_validation_dir))

In [ ]:
# ROI weights written by two_bit_optimization.ipynb (use_roi=True, roi_dataset='2s').
# ./best-model/ is not used any more - clean.sh deletes it.
base_dir = 'smart-pixels/weights/dataset_2s_roi_weights/'

fingerprint_choices = {
    # non-quantized models
    'Conv2D_Max'  : '392456de',
    'Conv2D_Full' : '392456de',
    'Conv2D_Slim' : '392456de',
    'Conv1D_Full' : '3eb13b90',
    'Conv1D_Slim' : '3eb13b90',
    'Mlp_Full'    : '392456de',
    'Mlp_Slim'    : '392456de',
    # quantized models
    'QConv2D_Max'  : '392456de',
    'QConv2D_Full' : '392456de',
    'QConv2D_Slim' : '392456de',
    'QConv1D_Full' : '3eb13b90',
    'QConv1D_Slim' : '3eb13b90',
    'QMlp_Full'    : '392456de',
    'QMlp_Slim'    : '392456de'
};

# Inventory of what has actually been trained, rather than an error per missing model
available = [k for k in fingerprint_choices
             if os.path.exists(base_dir + f'weights-2t-{k}-2bit_optimized-'
                               f'{fingerprint_choices[k]}-checkpoints/best_model.hdf5')]
print(f"models available in {base_dir}:")
for k in available:
    print(f"  {k}{'   <- selected' if k == model_type else ''}")

if model_type not in available:
    raise FileNotFoundError(
        f"{model_type} has not been trained here. Run two_bit_optimization.ipynb with "
        f"model_type='{model_type}' (and use_roi=True, roi_dataset='2s') first. "
        f"Available: {available}")

fingerprint = fingerprint_choices[model_type]

In [ ]:
weights_dir = base_dir + f'weights-2t-{model_type}-2bit_optimized-{fingerprint}-checkpoints'
best_model_hdf5 = f"{weights_dir}/best_model.hdf5"
best_model_keras = f"{weights_dir}/best_model.keras"
best_model_weights_hdf5 = f"{weights_dir}/best_model_weights.hdf5"
best_model_weights_keras = f"{weights_dir}/best_model_weights.keras"
model_architecture_json = f"{weights_dir}/model_architecture.json"

In [ ]:
print(best_model_hdf5)

## Load data

In [ ]:
# save_data_as_npy() writes npy/{timeslices}t-{model_type}-{fingerprint}-{X,y}_test.npy
# (two_bit_optimization.ipynb, 'Model Export & Utilities')
X = np.load(f'{npy_dir_val}/2t-{model_type}-{fingerprint}-X_test.npy')
y = np.load(f'{npy_dir_val}/2t-{model_type}-{fingerprint}-y_test.npy')

for _p in (f'{npy_dir_val}/2t-{model_type}-{fingerprint}-X_test.npy',
           f'{npy_dir_val}/2t-{model_type}-{fingerprint}-y_test.npy'):
    assert os.path.exists(_p), (f'{_p} missing - rerun two_bit_optimization.ipynb with '
                                f"model_type='{model_type}' to regenerate it")

In [ ]:
print(X.shape)
print(y.shape)

### Visualize data

Data visualization can also be moved to a different notebook, which may also already exists.

In [ ]:
# Plot some events
for i in range(4):
    plot_event(X, range(2), i)

In [ ]:
ani = animate_event(X, range(X.shape[3]), 0)
ani.save("animation.gif", writer="imagemagick", fps=2)
HTML(ani.to_jshtml())

### Quantize data

<b style="background-color: yellow; color: red">ATTENTION: Input quantization is disabled. Data is already quantized.</i></b>

## QKeras model

### Load QKeras model

In [ ]:
# Load the whole model from HDF5 file
from tensorflow.keras.models import load_model
from qkeras.utils import _add_supported_quantized_objects

custom_loss_choices = {
    # non-quantized models
    'Conv2D_Max'  : {"custom_loss": custom_loss},
    'Conv2D_Full' : {"custom_diag_loss": custom_diag_loss},
    'Conv2D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'Conv1D_Full' : {"custom_diag_loss": custom_diag_loss},
    'Conv1D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'Mlp_Full'    : {"custom_diag_loss": custom_diag_loss},
    'Mlp_Slim'    : {"custom_sse_loss": custom_sse_loss},
    # quantized models
    'QConv2D_Max'  : {"custom_loss": custom_loss},
    'QConv2D_Full' : {"custom_diag_loss": custom_diag_loss},
    'QConv2D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'QConv1D_Full' : {"custom_diag_loss": custom_diag_loss},
    'QConv1D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'QMlp_Full'    : {"custom_diag_loss": custom_diag_loss},
    'QMlp_Slim'    : {"custom_sse_loss": custom_sse_loss},
};

co = custom_loss_choices[model_type]
_add_supported_quantized_objects(co)
model = load_model(best_model_hdf5, custom_objects=co)
model.summary(line_length=120, show_trainable=True)

### Run QKeras model

In [ ]:
print(f'Input batch: {X.shape[0]}')
print(f'Input shape: {X.shape}')

#### QKeras Prediction

In [ ]:
y_qkeras = model.predict(np.ascontiguousarray(X))
print(f'QKeras output batch: {y_qkeras.shape[0]}')
print(f'QKeras output shape: {y_qkeras.shape}')

#### QKeras Profiling

In [ ]:
trace_qkeras = hls4ml.model.profiling.get_ymodel_keras(model, X)

In [ ]:
for key in trace_qkeras.keys():
    print(f'QKeras layer trace shape: {key} {trace_qkeras[key].shape}')

#### Save .dat files

In [ ]:
# Save input features and model predictions
np.savetxt(f"tb_input_features.dat", X.reshape(X.shape[0], -1), fmt="%f")
np.savetxt(f"tb_output_predictions.dat", y_qkeras.reshape(X.shape[0], -1), fmt="%f")

In [ ]:
!wc -l *tb_input_features.dat
!wc -l *tb_output_predictions.dat

## hls4ml model

### Configure hls4ml model

In [ ]:
project_name = f"SmtPxl_{model_type}_Vitis"

strategy = "Latency"
io_type = "io_parallel"
programmable_weights = False
# xcku115-flvb2104-2-e (plain Kintex UltraScale) is not installed in this
# machine's Vitis 2024.2 toolchain - only Kintex UltraScale+ (kintexuplus) is.
# xcku19p-ffvb2104-2-e is the largest installed KU+ part, keeping the same
# package code (2104) as originally intended. Verified with `set_part` in
# vitis_hls 2024.2 directly.
part = "xc7z020clg400-1"

hls4ml_output_dir = f"{project_name}_{io_type}_{strategy}{'_pw' if programmable_weights else ''}_hls4ml_prj"


In [ ]:
# Remove existing project
!rm -rf $hls4ml_output_dir*

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(
    model,
    granularity="name",
    # ap_fixed<16,7>, not the original <16,6>: same total width, but 1 more
    # integer bit / 1 fewer fractional bit. The original <16,6> was overflowing
    # (wrapping, not just losing resolution) in the QDense accumulators - going
    # from 6 to 7 integer bits fixes it, confirmed by parity-plot MSE dropping
    # ~24x for zero extra bits. Widening total width further (e.g. <18,7>)
    # helps a bit more but isn't needed to pass the parity plots.
    default_precision="ap_fixed<16,7>",
    default_reuse_factor=1
)

hls_config["Model"]["Strategy"] = strategy

if programmable_weights:
    # All the weights are exposed through BRAMs
    hls_config["Model"]["BramFactor"] = 0

#print(json.dumps(hls_config, indent=4))


<b style="background-color: yellow; color: red">ATTENTION: The model precision can be fine tuned here.</b>

#### Bit-precision

In [ ]:
print(model_type)

Catapult (`ac_fixed`) to Vitis (`ap_fixed`/`ap_ufixed`) precision syntax:

| Catapult | Vitis | Notes |
|---|---|---|
| `ac_fixed<W,I,true>` | `ap_fixed<W,I>` | signed by default in `ap_fixed` |
| `ac_fixed<W,I,false>` / `ufixed<W,I>` | `ap_ufixed<W,I>` | |
| `fixed<W,I>` (Catapult shorthand, signed) | `ap_fixed<W,I>` | |


In [ ]:
match model_type:
    case 'Conv2D_Max' | 'Conv2D_Full' | 'Conv2D_Slim' | 'Conv1D_Full' | 'Conv1D_Slim' | 'Mlp_Full'  | 'Mlp_Slim' :
        # Kind of unsopported, we care about QKeras models instead, see below
        pass
    case 'QConv2D_Max'  :
        hls_config['LayerName']['input_1']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['q_dense_2']['Precision']['result'] = 'ap_fixed<10,4>'
        hls_config['LayerName']['q_dense_2_linear']['Precision']['result'] = 'ap_fixed<10,4>'
    case 'QConv2D_Full' :
        hls_config['LayerName']['input_1']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['q_dense_2']['Precision']['result'] = 'ap_fixed<10,4>'
        hls_config['LayerName']['q_dense_2_linear']['Precision']['result'] = 'ap_fixed<10,4>'
    case 'QConv2D_Slim' :
        hls_config['LayerName']['input_1']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['q_dense_2']['Precision']['result'] = 'ap_fixed<10,4>'
        hls_config['LayerName']['q_dense_2_linear']['Precision']['result'] = 'ap_fixed<10,4>'
    case 'QConv1D_Full' :
        hls_config['LayerName']['input_pxls']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['avg_pooling_2d_proj_x']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['avg_pooling_2d_proj_y']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['dense_3']['Precision']['result'] = 'ap_fixed<10,4>'
        hls_config['LayerName']['dense_3_linear']['Precision']['result'] = 'ap_fixed<10,4>'
    case 'QConv1D_Slim' :
        hls_config['LayerName']['input_pxls']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['avg_pooling_2d_proj_x']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['avg_pooling_2d_proj_y']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['dense_3']['Precision']['result'] = 'ap_fixed<10,2>'
        hls_config['LayerName']['dense_3_linear']['Precision']['result'] = 'ap_fixed<10,2>'
    case 'QMlp_Full'    :
        hls_config['LayerName']['input_pxls']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['average_pooling2d_1']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['dense_3']['Precision']['result'] = 'ap_fixed<10,4>'
        hls_config['LayerName']['dense_3_linear']['Precision']['result'] = 'ap_fixed<10,4>'
    case 'QMlp_Slim'    :
        hls_config['LayerName']['input_pxls']['Precision']['result'] = 'ap_ufixed<2,2>'
        hls_config['LayerName']['average_pooling2d']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['average_pooling2d_1']['Precision']['result'] = 'ap_fixed<16,8>'
        hls_config['LayerName']['dense_3']['Precision']['result'] = 'ap_fixed<10,4>'
        hls_config['LayerName']['dense_3_linear']['Precision']['result'] = 'ap_fixed<10,4>'
    case _:
        pass


In [ ]:
# TODO: is this necessary?
with open(f"myproject_config_vitis.yml", "w") as yaml_file:
    yaml.dump(hls_config, yaml_file, explicit_start=False, default_flow_style=False)


In [ ]:
# Enable tracing for all of the layers
for layer in hls_config["LayerName"].keys():
    print("Enable tracing for layer:", layer)
    hls_config["LayerName"][layer]["Trace"] = True


In [ ]:
# Convert QKeras model to Vitis HLS C++
hls_model_hls4ml = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    output_dir=hls4ml_output_dir,
    project_name=project_name,
    part=part,
    backend="Vitis",
    io_type=io_type,
    clock_period=20,
    input_data_tb="tb_input_features.dat",
    output_data_tb="tb_output_predictions.dat",
)


In [ ]:
hls4ml.utils.plot_model(hls_model_hls4ml, show_shapes=True, show_precision=True, to_file=f"qkeras_vitis.png")
hls4ml.utils.plot_model(hls_model_hls4ml, show_shapes=True, show_precision=True, to_file=None)


In [ ]:
# Writing HLS project
hls_model_hls4ml.compile()

### Run hls4ml model

#### hls4ml Prediction

In [ ]:
y_hls4ml = hls_model_hls4ml.predict(np.ascontiguousarray(X))

In [ ]:
print(f'hls4ml output batch: {y_hls4ml.shape[0]}')
print(f'hls4ml output shape: {y_hls4ml.shape}')

#### hls4ml Profiling

In [ ]:
# Run tracing on the test set for the hls4ml model (fixed-point precision) 
pred_hls4ml, trace_hls4ml = hls_model_hls4ml.trace(X)

## Compare QKeras and hls4ml

### Trace visual inspection

### MSE per layer

In [ ]:
print(model_type)

In [ ]:
def mse(actual, predicted):
    return ((actual - predicted) ** 2).mean()

# print(trace_hls4ml.keys())
# print(trace_hls4ml['q_separable_conv2d_pointwise'].shape)
# print(trace_qkeras['q_separable_conv2d'].shape)
 
for key in trace_hls4ml.keys():
    if key == "q_separable_conv2d_depthwise":
        continue
    print("-------")
    print("MSE {} {}".format(
        key.replace("_pointwise", ""),
        mse(
            trace_hls4ml[key].flatten(), 
            trace_qkeras[key.replace("_pointwise", "")].flatten())))

### Parity plots

We use [parity plots](https://en.wikipedia.org/wiki/Parity_plot) to compare _hls4ml_ and QKeras implementations of each layer.
- If the _hls4ml_ implementation is accurate (i.e. matches QKeras implementation), the ouput values should lie close to the diagonal line `y=x`, indicating equivalence.
- Deviations from this line highlight inaccuracies or implementation errors.

In [ ]:
plot_layer_comparisons(trace_hls4ml, trace_qkeras, save_path="comparison_qkeras_hls4ml_vitis.jpeg")


### Performance Variables

In [ ]:
print(model_type)

In [ ]:
import os
import pandas as pd

# Same directory two_bit_optimization.ipynb writes its QKeras results to
# (performance_directory_2s_roi), so both land side by side. The filename carries a
# -hls4ml- infix, so it cannot collide with the QKeras "-vars.parquet".
vars_dir = (
    "./smart-pixels-ml/processed_parquets/"
    "dataset_2s_roi/"
    "test_dataset_2s_48x192_50x12P5_roi/2bit_optimized"
)

def save_performance_variables(df: pd.DataFrame, model_type: str, fingerprint: str, backend: str = "vitis") -> None:
    os.makedirs(vars_dir, exist_ok=True)
    vars_file = "{}/2t-{}-2bit_optimized-{}-hls4ml-{}-vars.parquet".format(
        vars_dir, model_type, fingerprint, backend
    )
    print(vars_file)
    df.to_parquet(vars_file)

if model_type in ('QConv2D_Max',):
    # True values/Labels (QKeras)
    df = pd.DataFrame(
        y, 
        columns=[
            'xtrue',
            'ytrue',
            'cotAtrue',
            'cotBtrue',
        ]
    )

    # Predicted values (hls4ml)
    df['x']                = y_hls4ml[:, 0]
    df['M11']              = y_hls4ml[:, 1]
    df['y']                = y_hls4ml[:, 2]
    df['M22']              = y_hls4ml[:, 3]
    df['cotA']             = y_hls4ml[:, 4]
    df['M33']              = y_hls4ml[:, 5]
    df['cotB']             = y_hls4ml[:, 6]
    df['M44']              = y_hls4ml[:, 7]
    df['M21']              = y_hls4ml[:, 8]
    df['M31']              = y_hls4ml[:, 9]
    df['M32']              = y_hls4ml[:,10]
    df['M41']              = y_hls4ml[:,11]
    df['M42']              = y_hls4ml[:,12]
    df['M43']              = y_hls4ml[:,13]

    # Residuals (true - predicted) WITH PARENTHESES
    df['residuals_x']                 = df['xtrue']                - df['x']
    df['residuals_y']                 = df['ytrue']                - df['y']
    df['residuals_cotA']              = df['cotAtrue']             - df['cotA']
    df['residuals_cotB']              = df['cotBtrue']             - df['cotB']

    # Sigmas
    for m in ["M11","M22","M33","M44"]:
            df[m] = 1e-9 + tf.math.maximum(df[m], 0.0)
    
    df["sigmax"]           = abs(df["M11"])
    df["sigmay"]           = np.sqrt(df["M21"]**2 + df["M22"]**2)
    df["sigmacotA"]        = np.sqrt(df["M31"]**2 + df["M32"]**2 + df["M33"]**2)
    df["sigmacotB"]        = np.sqrt(df["M41"]**2 + df["M42"]**2 + df["M43"]**2 + df["M44"]**2)

    save_performance_variables(df, model_type, fingerprint)

elif model_type in ('QConv2D_Full', 'QConv1D_Full', 'QMlp_Full'):
    # True values/Labels (QKeras)
    df = pd.DataFrame(
        y,
        columns=[
            'xtrue',
            'ytrue',
            'cotAtrue',
            'cotBtrue'
        ]
    )

    # Predicted values (hls4ml)
    df['x']         = y_hls4ml[:, 0]
    df['y']         = y_hls4ml[:, 1]
    df['cotA']      = y_hls4ml[:, 2]
    df['cotB']      = y_hls4ml[:, 3]
    df['M11']       = y_hls4ml[:, 4]
    df['M22']       = y_hls4ml[:, 5]
    df['M33']       = y_hls4ml[:, 6]
    df['M44']       = y_hls4ml[:, 7]

    # Residuals
    df['residuals_x']         = df['xtrue']         - df['x']
    df['residuals_y']         = df['ytrue']         - df['y']
    df['residuals_cotA']      = df['cotAtrue']      - df['cotA']
    df['residuals_cotB']      = df['cotBtrue']      - df['cotB']

    # Sigmas
    mapping = {
            'M11': 'sigmax',
            'M22': 'sigmay',
            'M33': 'sigmacotA',
            'M44': 'sigmacotB',
        }
    for m, new_name in mapping.items():
        df[new_name] = tf.nn.softplus(df[m]) + 1e-9

    save_performance_variables(df, model_type, fingerprint)


elif model_type in ('QConv2D_Slim', 'QConv1D_Slim', 'QMlp_Slim'):
    # True values/Labels (QKeras)
    df = pd.DataFrame(
        y, 
        columns=[
            'xtrue',
            'ytrue',
            'cotBtrue'
        ]
    )

    # Predicted values (hls4ml)
    df['x']        = y_hls4ml[:, 0]
    df['y']        = y_hls4ml[:, 1]
    df['cotB']     = y_hls4ml[:, 2]

    # Residuals
    df['residuals_x']        = df['xtrue']    - df['x']
    df['residuals_y']        = df['ytrue']    - df['y']
    df['residuals_cotB']     = df['cotBtrue'] - df['cotB']

    save_performance_variables(df, model_type, fingerprint)

else:
    pass


## Model synthesis

In [ ]:
print(f'Vitis hls4ml project: {os.getcwd()}/{hls4ml_output_dir}')


<b style="background-color: yellow; color: red">ATTENTION: Run Vitis HLS from the command line</i></b>

`build_prj.tcl` does not read command-line arguments - the stages it runs (csim/synth/cosim/validation/export/vsynth) are hardcoded in `build_opt.tcl` inside the project directory. Edit that file first if you want different stages than its defaults (e.g. set `cosim`/`validation` to `0` to skip the slow RTL cosimulation), then:

```
cd SmtPxl_MODEL_TYPE_Vitis_io_parallel_Latency_pw_hls4ml_prj
vitis_hls -f build_prj.tcl
```
</cell id="b73a9ccd">


### Results